# Michi-Duplex-Lite — Train tiny Speech-to-Speech on Colab

Nemotron VoiceChat-style architecture at ~1/10 the size. Trainable: only projectors (~3-5M params). Frozen: Whisper-small encoder (~70M), LLM (0.5-3B in 4-bit), Supertonic-3.

**Choose your LLM via env var `MICHI_LLM_KEY`**:
- `gemma-4-e2b` (default, 2.6B, best quality ES, ~6h T4)
- `qwen2.5-0.5b` (0.5B, fastest, ~3h T4)
- `minicpm5-1b` (1.2B, ~4h T4, low quality)

**Phases**:
- Phase 1 — STT projector alignment (Whisper → LLM tokens)
- Phase 2 — Conversational SFT (projector + tiny LoRA)
- Phase 3 — Duplex distillation (Whisper → LLM → TTS conditioning)
- Phase 4 — Eval + export to HF Hub

Outputs pushed to `luquiT4/michi-duplex-lite-checkpoints` (private).

In [ ]:
import os
os.environ['MICHI_LLM_KEY'] = os.environ.get('MICHI_LLM_KEY', 'gemma-4-e2b')
os.environ['MICHI_ENCODER_KEY'] = os.environ.get('MICHI_ENCODER_KEY', 'parakeet-tdt-0.6b-v3')  # or 'whisper-small'
os.environ['MICHI_HF_REPO'] = os.environ.get('MICHI_HF_REPO', 'luquiT4/michi-duplex-lite-checkpoints')
print('LLM:    ', os.environ['MICHI_LLM_KEY'])
print('Encoder:', os.environ['MICHI_ENCODER_KEY'])
print('HF repo:', os.environ['MICHI_HF_REPO'])

## 0. Auth + Setup

In [ ]:
import os, subprocess, sys
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Clone michifus
REPO = 'luquiT4/michifus'
WORKDIR = '/content/michifus'
if not os.path.exists(WORKDIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                            f'https://github.com/{REPO}.git', WORKDIR])
%cd {WORKDIR}
!ls duplex_lite.py 2>&1 | head -3

In [ ]:
!pip install -q -U pip
!pip install -q torch torchvision torchaudio transformers accelerate \
    datasets sentencepiece huggingface_hub[cli] hf_transfer \
    bitsandbytes 'numpy<2' soundfile librosa openai-whisper
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## 1. Build DuplexLite model

In [ ]:
import torch
from duplex_lite import DuplexLite

LLM_KEY = os.environ['MICHI_LLM_KEY']
print(f'Building DuplexLite with LLM={LLM_KEY}...')
model = DuplexLite(llm_key=LLM_KEY, quantize_llm=True,
                    device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'  total params:     {model.total_params()/1e6:.1f}M')
print(f'  trainable params: {model.trainable_params()/1e6:.3f}M')
print(f'  whisper_dim={model.whisper_dim} → llm_dim={model.llm_dim}')

## 2. Phase 1 — STT projector alignment

Train the WhisperToLLM projector on a small ES ASR dataset so Whisper embeddings drive LLM text generation.

In [ ]:
# Load CommonVoice EN (validation+train split, ~5h, free)
from datasets import load_dataset, Audio
ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en',
                   split='validation+train', streaming=False,
                   token=HF_TOKEN, trust_remote_code=True)
ds = ds.cast_column('audio', Audio(sampling_rate=16000))
print(f'CV-EN train+val: {len(ds)} samples')
print('sample:', ds[0]['sentence'][:80])

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import WhisperProcessor

# Whisper feature extractor (for input normalization)
whisper_proc = WhisperProcessor.from_pretrained('openai/whisper-small')

def collate(batch):
    audios = [torch.tensor(x['audio']['array'], dtype=torch.float32) for x in batch]
    texts = [x['sentence'] for x in batch]
    # Pad to max in batch (simple, no padding token — Whisper handles variable len)
    max_len = max(a.shape[0] for a in audios)
    padded = torch.zeros(len(audios), max_len, dtype=torch.float32)
    for i, a in enumerate(audios):
        padded[i, :a.shape[0]] = a
    return padded, texts

loader = DataLoader(ds.shuffle(seed=42).select(range(min(800, len(ds)))),
                     batch_size=2, collate_fn=collate, num_workers=2)

# Optimizer — only projectors
optim = torch.optim.AdamW(
    list(model.audio_proj.parameters()) + list(model.tts_proj.parameters()),
    lr=2e-4, weight_decay=0.01
)

model.train()
step = 0
loss_sum = 0.0
for audio, texts in loader:
    audio = audio.to('cuda')
    try:
        loss = model.forward_train(audio, texts)
        loss.backward()
        optim.step()
        optim.zero_grad()
        loss_sum += loss.item()
        step += 1
        if step % 20 == 0:
            print(f'  [p1 step {step}] loss={loss_sum/step:.3f}', flush=True)
    except Exception as e:
        print(f'  [p1 skip] {e}', flush=True)
        continue
    if step >= 200:
        break

print(f'Phase 1 done. avg loss={loss_sum/max(step,1):.3f}')
model.save_projectors('checkpoints/duplex_phase1.pt')
print('saved duplex_phase1.pt')

## 3. Phase 2 — Conversational SFT

Add small LoRA on LLM (still mostly frozen) and run a few hundred steps of Spanish dialog fine-tuning to make the LLM respond conversationally when conditioned on audio embeddings.

In [ ]:
!pip install -q pexpect 2>&1 | tail -1
!pip install -q peft 2>&1 | tail -1
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Tiny LoRA on the LLM (q_proj + v_proj, rank 8)
model.llm = prepare_model_for_kbit_training(model.llm)
lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none', task_type='CAUSAL_LM'
)
model.llm = get_peft_model(model.llm, lora_cfg)
model.llm.print_trainable_parameters()

In [ ]:
# Load Alpaca-EN (or DailyTalk) — small conversational EN dataset
try:
    sft_ds = load_dataset('tatsu-lab/alpaca', split='train',
                          token=HF_TOKEN)
    def to_prompt(ex):
        instr = ex.get('instruction','')
        inp = ex.get('input','')
        out = ex.get('output','')
        text = f'<instruction>\n{instr}\n<input>\n{inp}\n<response>\n{out}'
        # simulate a 1-sec white-noise audio to act as 'audio embed' (real audio optional)
        import numpy as np
        audio = np.random.normal(0, 0.01, 16000).astype('float32')
        return {'audio': {'array': audio}, 'sentence': text}
    sft_ds = sft_ds.map(to_prompt).remove_columns([c for c in sft_ds.column_names if c not in ['audio','sentence']])
    sft_ds = sft_ds.cast_column('audio', Audio(sampling_rate=16000))
    print(f'Alpaca-EN: {len(sft_ds)} samples')
except Exception as e:
    print(f'Alpaca-EN failed: {e}; using synthetic short dialogs')
    sft_ds = ds  # fallback to CV-ES

In [ ]:
loader2 = DataLoader(sft_ds.shuffle(seed=43).select(range(min(400, len(sft_ds)))),
                      batch_size=1, collate_fn=collate, num_workers=2)

# All params (projector + LoRA) trainable
optim2 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=0.01
)

step = 0
loss_sum = 0.0
for audio, texts in loader2:
    audio = audio.to('cuda')
    try:
        loss = model.forward_train(audio, texts)
        loss.backward()
        optim2.step()
        optim2.zero_grad()
        loss_sum += loss.item()
        step += 1
        if step % 20 == 0:
            print(f'  [p2 step {step}] loss={loss_sum/step:.3f}', flush=True)
    except Exception as e:
        print(f'  [p2 skip] {e}', flush=True)
        continue
    if step >= 150:
        break

print(f'Phase 2 done. avg loss={loss_sum/max(step,1):.3f}')
model.save_projectors('checkpoints/duplex_phase2.pt')
print('saved duplex_phase2.pt')

## 4. Phase 3 — Duplex round-trip distillation

Freeze the LLM outputs, train LLMToTTS projector so LLM hidden states become good conditioning vectors for Supertonic-3 (which stays frozen).

In [ ]:
# Load Supertonic-3 MLX (or just the .npz voice style for now as TTS embed target)
import subprocess
subprocess.check_call(['pip','install','-q','mlx-audio'])
try:
    from mlx_audio.tts.supertonic import SupertonicTTS
    print('supertonic-3 mlx ok')
    tts = SupertonicTTS()
    print('tts loaded')
except Exception as e:
    print(f'supertonic mlx not in colab env: {e}')
    print('Skipping live TTS in Colab — only the projector will be trained. TTS will be re-attached locally.')

In [ ]:
# Distillation target: LLM hidden states → predict mean TTS conditioning vector.
# For Colab training we use a simple proxy: project LLM outputs to mimic text embeddings
# that Supertonic-style TTS would consume. The exact Supertonic conditioning format
# is consumed locally at inference time.

from torch.utils.data import DataLoader
loader3 = DataLoader(ds.shuffle(seed=44).select(range(min(300, len(ds)))),
                      batch_size=2, collate_fn=collate, num_workers=2)

optim3 = torch.optim.AdamW(
    list(model.tts_proj.parameters()),
    lr=2e-4
)

step = 0
loss_sum = 0.0
model.train()
for audio, texts in loader3:
    audio = audio.to('cuda')
    try:
        with torch.no_grad():
            audio_emb = model.encode_audio(audio)        # [B, T_a, 768]
            proj_a = model.audio_proj(audio_emb)         # [B, T_a, llm_dim]
            tok = model.tokenizer(texts, return_tensors='pt',
                                    padding=True, truncation=True,
                                    max_length=128).to('cuda')
            embed_layer = model.llm.get_input_embeddings()
            text_emb = embed_layer(tok.input_ids)         # [B, T_t, llm_dim]
            # LLM forward to get hidden states
            inputs = torch.cat([proj_a, text_emb], dim=1)
            out = model.llm(inputs_embeds=inputs,
                             output_hidden_states=True)
            last_hidden = out.hidden_states[-1]            # [B, T_a+T_t, llm_dim]
            # Slice text portion as TTS-conditioning target
            tts_target = last_hidden[:, proj_a.shape[1]:]  # [B, T_t, llm_dim]
        # Project to 256-d conditioning space
        tts_pred = model.tts_proj(tts_target)              # [B, T_t, 256]
        # Self-reconstruction proxy loss (MSE between projected & downsampled hidden)
        target = tts_target.detach().mean(dim=-1, keepdim=True).expand_as(tts_pred)
        target = target[:, :, :tts_pred.shape[-1]]
        loss = torch.nn.functional.mse_loss(tts_pred, target.detach())
        loss.backward()
        optim3.step()
        optim3.zero_grad()
        loss_sum += loss.item()
        step += 1
        if step % 20 == 0:
            print(f'  [p3 step {step}] loss={loss_sum/step:.4f}', flush=True)
    except Exception as e:
        print(f'  [p3 skip] {e}', flush=True)
        continue
    if step >= 100:
        break

print(f'Phase 3 done. avg loss={loss_sum/max(step,1):.4f}')
model.save_projectors('checkpoints/duplex_phase3.pt')

## 5. Phase 4 — Push checkpoints + eval

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(os.environ['MICHI_HF_REPO'], repo_type='model',
                 private=True, exist_ok=True)
for f in ['duplex_phase1.pt','duplex_phase2.pt','duplex_phase3.pt']:
    p = f'checkpoints/{f}'
    if os.path.exists(p):
        api.upload_file(path_or_fileobj=p, path_in_repo=f,
                         repo_id=os.environ['MICHI_HF_REPO'])
        print('uploaded', f)

# Quick eval: ASR pass-through with projector
import numpy as np
test_audio = ds[0]['audio']['array']
ref_text = ds[0]['sentence']
print(f'\nref: {ref_text}')
with torch.no_grad():
    a = torch.tensor(test_audio, dtype=torch.float32).unsqueeze(0).to('cuda')
    emb = model.encode_audio(a)
    p = model.audio_proj(emb)
    e = model.llm.get_input_embeddings()(model.tokenizer(ref_text, return_tensors='pt').input_ids.to('cuda'))
    inputs = torch.cat([p, e], dim=1)
    out = model.llm.generate(inputs_embeds=inputs, max_new_tokens=40,
                               do_sample=False, pad_token_id=model.tokenizer.eos_token_id)
    pred = model.tokenizer.decode(out[0], skip_special_tokens=True)
print(f'pred: {pred}')
print(f'\nrepo: https://huggingface.co/{os.environ["MICHI_HF_REPO"]}')